In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [2]:
# ==========================================
# Your existing data
# X 8D (40,8)
# y 1D (40,)
# ==========================================

# Example
X = np.load('function_8/initial_inputs.npy')
y = np.load('function_8/initial_outputs.npy')


X = np.vstack([X, [0.128005, 0.154897, 0.058272, 0.122000, 0.986900, 0.039907, 0.103405, 0.976752],
               [0.203077, 0.196661, 0.153981, 0.203857, 0.986902, 0.910406, 0.335567, 0.984300],
               [0.009077, 0.417644, 0.022929, 0.199816, 0.986902, 0.871712, 0.292497, 0.897838],
               [0.009077, 0.003419, 0.022929, 0.009043, 0.986902, 0.024766, 0.341373, 0.123022],
               [0.009077, 0.345791, 0.022929, 0.009043, 0.009649, 0.961092, 0.035909, 0.976885],
               [0.221973, 0.099559, 0.022929, 0.370122, 0.986902, 0.029949, 0.228365, 0.988755],
               [0.091349, 0.098277, 0.190682, 0.039787, 0.986902, 0.504223, 0.195190, 0.977689],
               [0.009077, 0.003419, 0.070977, 0.484072, 0.986902, 0.536657, 0.046557, 0.908638],
               [0.139339, 0.162429, 0.125695, 0.230561, 0.721600, 0.413704, 0.174870, 0.439044],
               [0.028831, 0.041562, 0.347374, 0.049758, 0.832138, 0.819064, 0.038738, 0.634959],
               [0.107867, 0.164186, 0.141784, 0.148790, 0.773518, 0.496458, 0.199460, 0.144128]
            ])

y = np.append(y, [9.7201819727396,
                  9.734522438073,
                  9.6933403472386,
                  9.6016815685446,
                  9.29788130054,
                  9.629718275198499,
                  9.9421860481249,
                  9.7644996173566,
                  9.9758307355114,
                  9.6718569807679,
                  9.9781112207936
])

In [3]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=np.ones(X.shape[1]), nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(10000):

    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Next point to evaluate:
[0.115171 0.157412 0.14406  0.151372 0.78925  0.493003 0.197626 0.100847]
Expected Improvement:
0.0027093758173301935
